In [0]:
from pyspark.sql.functions import *

## Task 1:
Create a DataFrame from an in-memory Python list/dict and display it.

In [0]:
data = [
    {"name": "Kirit", "age": 22},
    {"name": "Manpreet", "age": 25},
    {"name": "Surajeet", "age":21}
]

df = spark.createDataFrame(data)
df.display()

## Task 2:
Read a CSV with header=True and inferSchema, then read the same file with an explicit StructType
schema; compare the two resulting schemas.

In [0]:
df_1 = (spark.read
    .format('csv')
    .option('header', 'true')
    .option('inferSchema', 'true')
    .load('/Volumes/cyntexa_dev/logistics/raw/logistics_2026_08_18.csv')
)

df_1.display()

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DateType, DoubleType

schema = StructType([
    StructField("shipment_id", StringType(), False),
    StructField("origin_city", StringType(), False),
    StructField("destination_city", StringType(), False),
    StructField("carrier", StringType(), False),
    StructField("status", StringType(), False),
    StructField("ship_date", DateType(), False),
    StructField("weight_kg", DoubleType(), False)
])

df_2 = (spark.read
        .format('csv')
        .schema(schema)
        .option("header", "true")
        .load('/Volumes/cyntexa_dev/logistics/raw/logistics_2026_08_18.csv'))

df_2.display()

In [0]:
df_1.printSchema()

df_2.printSchema()

In [0]:
print(df_1.schema == df_2.schema)

# The schema is identical but it still says false

## Task 3:
Apply one filter transformation followed by one action (count() or show()), and explain in your own
words why nothing ran until the action.

In [0]:
df_filtered = df_1.filter(col("weight_kg") < 7)

In [0]:
df_filtered.display()

Apache Spark uses the lazy evaluation for the transformation and actions defined in the dataframes.

So Spark optimizes the data processing (transformations) by identifing the most optimized physical plan, however spark does not work on the plan unitl the actions are called, rather evaluating each transformation in the order it waits until the actions trigger the computation on the transformation. This is called the **Lazy Evaluation**

## Task 4:
Build a small end-to-end ELT: read CSV, filter, add a column (e.g., ingestion_date), and write the
result as Delta.

In [0]:
df_1 = (spark.read
    .format('csv')
    .option('header', 'true')
    .option('inferSchema', 'true')
    .load('/Volumes/cyntexa_dev/logistics/raw/logistics_2026_08_18.csv')
)

(df_1
    .filter(col("weight_kg") < 7)
    .withColumn("ingestion_date", current_date())
    .write
    .format('delta')
    .mode("append")
    .saveAsTable('cyntexa_dev.logistics.less_than_7')
)

## Task 5:
Read a JSON file with nested structure and flatten at least one nested field using dot notation or
explode().

In [0]:
json_file = spark.read.option("multiline", "true").json('/Volumes/cyntexa_dev/logistics/raw/logistics_customers_2026_08_18.json')

In [0]:
#Using the dot notation

df_flattened = json_file.select(
    "shipment_id",
    "ship_date",
    "customer.name",
    "customer.contact.email",
    "customer.contact.phone",
    "items",
    "carrier",
    "total_weight_kg"
)

df_flattened.display()

But here also the array (items here) can't be divided by the dot notation so we will use the explode() function to divide the array.

In [0]:
df_flattened = json_file.select(
    "shipment_id",
    "ship_date",
    "carrier",
    "customer.name",
    "customer.contact.email",
    "customer.contact.phone",
    explode("items").alias("item"),
    "total_weight_kg"
).select(
    "shipment_id",
    "ship_date",
    "carrier",
    "name",
    "email",
    "phone",
    "item.qty",
    "item.sku",
    "total_weight_kg"
)

df_flattened.display()

## Task 6:
Call .explain() on a multi-step transformation chain and identify, from the physical plan, which steps got pipelined together versus which required a shuffle.

In [0]:
df_3 = (
    df_flattened
    .filter(col("total_weight_kg") < 7)
    .withColumn("ingestion_date", current_date())
    .select("shipment_id", "carrier", "qty", "sku", "total_weight_kg", "ingestion_date")
    .groupBy("carrier")
    .sum("total_weight_kg").alias("total_carrier_load")
)

df_3.explain()

Explanation?

## Task 7:
Take a pandas-based script (your own, or a sample provided by your instructor) and rewrite it in PySpark, documenting at least 3 places where the pandas approach would not scale and how Spark's approach solves it.

In [0]:
## +++++++++ PANDAS SCRIPT ++++++++++++++++++


import pandas as pd

# Sample shipment data
data = [
    {"shipment_id": "SHP2001", "carrier": "BlueDart", "weight": 12.5, "status": "Delivered"},
    {"shipment_id": "SHP2002", "carrier": "DTDC", "weight": 8.2, "status": "Pending"},
    {"shipment_id": "SHP2003", "carrier": "BlueDart", "weight": 20.0, "status": "Delivered"},
    {"shipment_id": "SHP2004", "carrier": "Delhivery", "weight": 5.4, "status": "Pending"},
    {"shipment_id": "SHP2005", "carrier": "DTDC", "weight": 15.7, "status": "Delivered"},
    {"shipment_id": "SHP2006", "carrier": "BlueDart", "weight": 3.3, "status": "Pending"},
    {"shipment_id": "SHP2007", "carrier": "Delhivery", "weight": 9.8, "status": "Delivered"},
    {"shipment_id": "SHP2008", "carrier": "DTDC", "weight": 2.1, "status": "Pending"},
]

# Create Pandas DataFrame
df_pd = pd.DataFrame(data)

# 1. Filter heavy shipments
heavy_shipments = df_pd[df_pd["weight"] > 10]

# 2. Calculate average weight by carrier
carrier_stats = (
    df_pd.groupby("carrier")["weight"]
      .mean()
      .reset_index()
      .rename(columns={"weight": "avg_weight"})
)

# 3. Count delivered shipments by carrier
delivered_count = (
    df_pd[df_pd["status"] == "Delivered"]
    .groupby("carrier")
    .size()
    .reset_index(name="delivered_shipments")
)

# 4. Merge the results
result = carrier_stats.merge(
    delivered_count,
    on="carrier",
    how="left"
)

# 5. Sort by average weight
result = result.sort_values(
    "avg_weight",
    ascending=False
)

print(result)

In [0]:
# ++++++++++++++++++++ PYSPARK SCRIPT ++++++++++++++++++++++

# from pyspark.sql.functions import *

data = [
    {"shipment_id": "SHP2001", "carrier": "BlueDart", "weight": 12.5, "status": "Delivered"},
    {"shipment_id": "SHP2002", "carrier": "DTDC", "weight": 8.2, "status": "Pending"},
    {"shipment_id": "SHP2003", "carrier": "BlueDart", "weight": 20.0, "status": "Delivered"},
    {"shipment_id": "SHP2004", "carrier": "Delhivery", "weight": 5.4, "status": "Pending"},
    {"shipment_id": "SHP2005", "carrier": "DTDC", "weight": 15.7, "status": "Delivered"},
    {"shipment_id": "SHP2006", "carrier": "BlueDart", "weight": 3.3, "status": "Pending"},
    {"shipment_id": "SHP2007", "carrier": "Delhivery", "weight": 9.8, "status": "Delivered"},
    {"shipment_id": "SHP2008", "carrier": "DTDC", "weight": 2.1, "status": "Pending"},
]

df_py = spark.createDataFrame(data)

heavy_shipments = df_py.filter(col("weight") > 10)

carrier_stats = (df_py
                .groupBy("carrier")
                .agg(
                    avg("weight").alias("avg_weight")
                ))

delivered_count = (df_py.filter(col("status") == "Delivered")
                   .groupBy("carrier")
                   .agg(count("*")).alias("total_delivered"))

results = carrier_stats.join(delivered_count, on="carrier", how="left").orderBy(col("avg_weight").desc())

results.display()

- **Filtering:** Pandas evaluate the filters eagerly whereas the spark waits for the actions to trigger the compute, so it just makes aefficient physical plan. But if the dataset is too large using pandas will be very expensive as it continously executes the code.
- **groupBy and agg :** Pandas performs the aggeregation on a single machine which can become the bottleneck for the large dataset, where as spark divides the aggergation among the partitions before shuffling data reducing the amount of data which is required to be transfered.
- **Join:** Pandas merge the dataFrame on the same machine which can quickly become bottle neck for the large datasets, but spark distibute among the workers and choose a appropriate join strategy.

## Task 8:


In [0]:
sales_df = spark.read.csv("/Volumes/cyntexa_dev/logistics/raw/sales_2026_08_18.csv", header=True, inferSchema=True)
cleaned_sales_df = sales_df.dropDuplicates().dropna(how="any")

In [0]:
cleaned_sales_df.write.mode("overwrite").partitionBy("order_date").saveAsTable("cyntexa_dev.bronze.sales_by_date")

partitionBy("order_date")` it partition the table based on order date, and we can check that in the `DESC DETAIL` command that it was partitionBy "order_date".
le at once spark does partiion pruning, and read onl the required part because I have partitioned by order_date so when I perfrm this quey
```
SELECT *
FROM dev.silver.sales_partition
WHERE order_date BETWEEN "2023-12-01" AND "2023-1202";
```
spark sql under the hood cecks the partitions that was required and not read the entire table, so the speed of the task is increased. So that's how it executes the physical plan.

## Task 9:
(Data Analyst) Using a Spark DataFrame (not SQL), reproduce a report you'd normally build in
Excel/pandas — e.g., monthly revenue by category — and export the result for a dashboard.

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, DateType

shipment_schema = StructType([
    StructField("shipment_id", StringType(), False),
    StructField("origin_city", StringType(), False),
    StructField("destination_city", StringType(), False),
    StructField("carrier", StringType(), False),
    StructField("status", StringType(), False),
    StructField("ship_date", DateType(), False),
    StructField("weight_kg", DoubleType(), False),
])

shipments_df = spark.read.option("header", "true").schema(shipment_schema).csv("/Volumes/cyntexa_dev/logistics/raw/logistics_2026_08_18.csv")

shipments_df.display()

In [0]:
enriched_df = shipments_df.withColumn(
    "weight_category",
    when(col("weight_kg") < 5, "light")
    .when(col("weight_kg") < 15, "medium")
    .otherwise("heavy")
)

enriched_df.display()

In [0]:
enriched_df.write.mode("overwrite").saveAsTable("cyntexa_dev.logistics.enriched_shipments")

I will use this table to be used in the dashboard.